# Convertir y validar

Convierte cada archivo mensual descargado en `0.0` a parquet y lo compara con los totales de control que publica la DIAN.

## Conversión

Cada zip trae un único `.xlsx` con una fila por **ítem de declaración** (una subpartida arancelaria dentro de una declaración de importación o exportación). Se guarda tal cual en `data/interim/<flujo>/<año>/<flujo>_<aaaamm>.parquet`, con dos decisiones:

- **Todas las columnas se guardan como texto, exactamente como las publica la DIAN.** En el Excel, muchos códigos están guardados como números, así que leerlos como números les borraría los ceros iniciales (la DIAN ya publica algunas subpartidas con 9 dígitos en vez de 10, y códigos como `COD_CLASE_IMPORTADOR = '02'`) o los convertiría en decimales (NIT, números de formulario). Asignar el tipo de cada columna es un paso posterior, explícito y documentado.
- **No se unifican columnas todavía.** Las columnas cambian a lo largo de los años. Por ejemplo, enero de 2001 tiene 46 columnas en exportaciones y 121 en importaciones; julio y agosto de 2026 tienen 72 y 162. Algunas se renombraron (`NIT_EXPORTADOR_` → `NIT_EXPORTADOR`) y otras no existían, como el nombre del exportador en 2001. El inventario de columnas de abajo registra qué columnas tiene cada mes, para unificarlas en el paso siguiente.

Solo se limpian los nombres de columna (se les quitan los espacios al principio y al final); los valores no se modifican.

Un mes ya convertido se saltea, salvo que su zip sea más nuevo que el parquet (una versión corregida que bajó `0.0`). Siempre se usa la versión más reciente de cada mes.

## Validación contra los totales de control

Para cada mes se compara, contra los totales de control de la DIAN:

| Medida | Importaciones | Exportaciones |
|---|---|---|
| Registros | cantidad de filas | cantidad de filas |
| CIF US$ | suma de `VALOR_CIF_USD` | — |
| FOB US$ | suma de `VALOR_FOB_USD` | suma de `VALOR_FOB_USD` |
| Peso neto (kg) | suma de `PESO_NETO` | suma de `PESO_NETO_KGS` |

La cantidad de registros tiene que coincidir exactamente. Las sumas pueden diferir por redondeo, ya que los valores del Excel tienen dos decimales; se acepta una diferencia relativa de hasta una millonésima. Por ejemplo, en importaciones de julio de 2026 el FOB suma 7.488.466.120,11 y el total de control dice 7.488.466.120,26: 15 centavos de diferencia sobre 7.488 millones. Si un mes no tiene la columna de alguna medida, esa comparación figura como fallida.

El resultado queda en `data/interim/validacion_totales_control.parquet`, una fila por mes y medida, y los meses que no coinciden se muestran abajo.

El código está en `colombia_dian_comex/pipeline.py` (`convert_month_to_parquet`, `read_control_totals`, `validate_month`, `validation_report`).

In [ ]:
import logging

import polars as pl
from tqdm.auto import tqdm

from colombia_dian_comex import utils
from colombia_dian_comex.pipeline import (
    Flow,
    convert_month_to_parquet,
    local_monthly_zips,
    validation_report,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

## Paso 1: convertir cada mes

Cada xlsx se lee completo en memoria (el más grande, de importaciones, ocupa ~310 MB y tarda ~30 segundos) y se escribe como parquet comprimido (~50 MB para un mes de importaciones de 2026).

In [ ]:
converted = {
    flow: {
        period: convert_month_to_parquet(flow, period, zip_path)
        for period, zip_path in tqdm(
            local_monthly_zips(flow).items(), desc=f"Convirtiendo {flow}"
        )
    }
    for flow in Flow
}

## Paso 2: validar contra los totales de control

In [ ]:
df_validation = pl.concat(
    [validation_report(flow, list(months)) for flow, months in converted.items()]
)
df_validation.write_parquet(
    utils.data_interim_dir("validacion_totales_control.parquet")
)

df_failed = df_validation.filter(~pl.col("OK"))
logging.info(
    "%d de %d comparaciones coinciden con los totales de control.",
    df_validation["OK"].sum(),
    df_validation.height,
)
df_failed

## Paso 3: inventario de columnas

Qué columnas tiene cada mes, en orden. Se guarda en `data/interim/inventario_columnas.parquet` (una fila por flujo, mes y columna) y sirve para documentar y diseñar la unificación de columnas entre años. La tabla de abajo agrupa los meses que comparten exactamente las mismas columnas.

In [ ]:
df_inventory = pl.DataFrame(
    [
        {
            "FLUJO": flow.value,
            "PERIODO": period,
            "POSICION": position,
            "COLUMNA": column,
        }
        for flow, months in converted.items()
        for period, path in months.items()
        for position, column in enumerate(
            pl.scan_parquet(path).collect_schema().names(), 1
        )
    ]
)
df_inventory.write_parquet(utils.data_interim_dir("inventario_columnas.parquet"))

# Distinct column layouts per flow, with the months that use each one.
(
    df_inventory.group_by("FLUJO", "PERIODO", maintain_order=True)
    .agg(
        pl.col("COLUMNA").str.join("|").alias("COLUMNAS"), pl.len().alias("N_COLUMNAS")
    )
    .group_by("FLUJO", "COLUMNAS", "N_COLUMNAS", maintain_order=True)
    .agg(
        pl.col("PERIODO").min().alias("DESDE"),
        pl.col("PERIODO").max().alias("HASTA"),
        pl.len().alias("MESES"),
    )
    .drop("COLUMNAS")
)